# 03 — Evaluating the recruitment matching engine

> **Applied Labs** · **Domain**: AG · **Difficulty**: Advanced · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/08_recruitment_matching/03_evaluate.ipynb)

**Prerequisites**:
- [02_build.ipynb](./02_build.ipynb)

**What you will learn**:
- How to build an evaluation framework with matching accuracy, consistency, fairness, and explanation quality
- Measuring rank correlation against expert judgments using Kendall τ and Spearman ρ
- Assessing rubric consistency via repeated scoring and inter-rater benchmarks
- Comprehensive fairness analysis with demographic parity, equal opportunity, and the 4/5ths rule
- LLM-as-judge evaluation of explanation quality
- Cost and efficiency analysis comparing AI to manual screening

In [ ]:
# @title Setup — Run this cell first
!pip install -q "openai>=1.0.0" "pandas>=2.0.0" "matplotlib>=3.7.0" "numpy>=1.24.0" "scipy>=1.10.0"

import os, json, textwrap
from dataclasses import dataclass, field, asdict
from typing import Optional, Dict, List, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kendalltau, spearmanr

from openai import OpenAI

client: Optional[OpenAI] = None
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    client = OpenAI(api_key=api_key)
    print("OpenAI client initialised ✓")
else:
    print("⚠ OPENAI_API_KEY not set — LLM-as-judge calls will use mock scores")

np.random.seed(42)
print("Setup complete ✓")

## Section 1 — Evaluation framework

We evaluate the recruitment matching engine across four dimensions:

| Dimension | Metrics | Why it matters |
|-----------|---------|----------------|
| **Matching accuracy** | Kendall τ, Spearman ρ, per-criterion accuracy | Do rankings agree with expert judgment? |
| **Rubric consistency** | Score variance, coefficient of variation | Same resume → same score? |
| **Fairness** | Demographic parity, equal opportunity, 4/5ths rule | No disparate impact on protected groups? |
| **Explanation quality** | Specificity, evidence-based, actionable, fair | Are explanations useful and unbiased? |

In [ ]:
@dataclass
class EvalConfig:
    """Configuration for evaluation runs."""
    pass_threshold: float = 3.0       # score >= threshold → "pass"
    consistency_runs: int = 3         # number of repeated evaluations
    fairness_alpha: float = 0.8       # 4/5ths rule threshold
    explanation_samples: int = 10     # explanations to evaluate

@dataclass
class EvalResult:
    """Result of a single evaluation dimension."""
    dimension: str
    metric_name: str
    value: float
    benchmark: float
    passes: bool
    detail: str = ""

config = EvalConfig()

# ── Simulated data from the build notebook ──
# Expert rankings (1 = best) for 10 candidates on "Data Scientist" role
CANDIDATES: List[str] = [
    "Alice Chen", "Bob Martinez", "Carol Okafor", "David Park",
    "Elena Volkov", "Frank Liu", "Grace Nakamura",
    "Hassan Al-Rashid", "Iris Thompson", "James Wright",
]

# Expert ranking (1 = best candidate for data scientist role)
EXPERT_RANKS: List[int] = [2, 1, 4, 9, 3, 7, 6, 5, 8, 10]

# System scores (from matching engine)
SYSTEM_SCORES: List[float] = [4.15, 4.35, 3.45, 2.10, 4.05, 2.85, 3.10, 3.55, 2.45, 1.95]

# Per-criterion scores (Technical, Experience, Education, Culture, Growth)
CRITERION_SCORES: Dict[str, List[int]] = {
    "Technical":  [4, 5, 3, 2, 4, 2, 3, 3, 2, 2],
    "Experience": [5, 4, 5, 3, 4, 4, 3, 5, 3, 1],
    "Education":  [4, 4, 5, 3, 4, 5, 3, 3, 3, 3],
    "Culture":    [4, 3, 4, 2, 3, 4, 3, 4, 3, 2],
    "Growth":     [3, 3, 4, 2, 4, 3, 3, 3, 2, 2],
}

# Background groups for fairness
GROUPS: List[str] = ["A", "A", "B", "A", "B", "A", "B", "B", "A", "A"]

# Explanations generated by the system
EXPLANATIONS: List[str] = [
    "Strong technical match: 5/6 requirements met including Python (5 yrs), "
    "ML (3 yrs), and SQL. Team lead experience demonstrates culture fit. "
    "Recommended for senior-level interview loop.",

    "Excellent ML fit: PyTorch expert with published NeurIPS paper. "
    "Deployed 12 production models with MLflow. Stanford MS Statistics "
    "provides strong foundation. Top candidate.",

    "Non-traditional background but strong analytical skills. PhD Physics "
    "with 12 publications shows research rigor. R/MATLAB experience "
    "translates to Python/ML. Recommend technical assessment.",

    "Frontend-focused background. React/Node.js stack does not align with "
    "backend ML requirements. Limited Python, no ML experience. "
    "Would need significant upskilling.",

    "Strong NLP and deep learning expertise. Deployed transformer models "
    "at scale (5M req/day). International education equivalent to MS. "
    "8 production systems demonstrate deployment capability.",

    "Product management background, not technical IC role. Strong data "
    "analysis skills but limited ML model development experience. "
    "Better fit for PM roles.",

    "Transitioning from data analysis to ML. Coursera specialisation "
    "shows initiative but limited production ML experience. "
    "Could be strong junior candidate.",

    "Systems architecture expert. Kubernetes and infrastructure skills "
    "are strong but ML experience is absent. Would need ML ramp-up. "
    "Platform engineering would be better fit.",

    "Design background transitioning to PM. Limited technical depth for "
    "data science role. User research skills valuable but insufficient "
    "for ML-heavy position.",

    "Career changer from teaching. Bootcamp training covers basics but "
    "1 year experience is insufficient for senior role. Mathematics "
    "background is relevant foundation.",
]

print("=" * 60)
print("  EVALUATION FRAMEWORK — DATA LOADED")
print("=" * 60)
print(f"  Candidates         : {len(CANDIDATES)}")
print(f"  Expert rankings    : {len(EXPERT_RANKS)} (1 = best)")
print(f"  System scores      : {len(SYSTEM_SCORES)} (1-5 scale)")
print(f"  Criterion dimensions: {len(CRITERION_SCORES)}")
print(f"  Background groups  : {dict((g, GROUPS.count(g)) for g in set(GROUPS))}")
print(f"  Explanations       : {len(EXPLANATIONS)}")
print(f"\n  Config: pass_threshold={config.pass_threshold}, "
      f"consistency_runs={config.consistency_runs}")
print("✓ Evaluation data ready")

## Section 2 — Matching accuracy

We compare the system's candidate rankings against expert human rankings using
rank correlation metrics. Higher correlation means the system agrees with expert
judgment about who the best candidates are.

In [ ]:
def evaluate_matching_accuracy(
    system_scores: List[float],
    expert_ranks: List[int],
    candidates: List[str],
) -> Dict[str, EvalResult]:
    """Evaluate matching accuracy using rank correlation."""
    # Convert scores to ranks (higher score = lower rank number)
    score_order = np.argsort(system_scores)[::-1]
    system_ranks = np.empty_like(score_order)
    system_ranks[score_order] = np.arange(1, len(system_scores) + 1)

    tau, tau_p = kendalltau(expert_ranks, system_ranks)
    rho, rho_p = spearmanr(expert_ranks, system_ranks)

    # Per-position accuracy: how many top-K overlap?
    expert_top3 = set(np.argsort(expert_ranks)[:3])
    system_top3 = set(score_order[:3])
    top3_overlap = len(expert_top3 & system_top3) / 3.0

    expert_top5 = set(np.argsort(expert_ranks)[:5])
    system_top5 = set(score_order[:5])
    top5_overlap = len(expert_top5 & system_top5) / 5.0

    return {
        "kendall_tau": EvalResult("Accuracy", "Kendall τ", round(tau, 3),
                                  0.6, tau >= 0.6,
                                  f"p={tau_p:.4f}"),
        "spearman_rho": EvalResult("Accuracy", "Spearman ρ", round(rho, 3),
                                   0.7, rho >= 0.7,
                                   f"p={rho_p:.4f}"),
        "top3_precision": EvalResult("Accuracy", "Top-3 precision", round(top3_overlap, 3),
                                     0.67, top3_overlap >= 0.67,
                                     f"{int(top3_overlap*3)}/3 overlap"),
        "top5_precision": EvalResult("Accuracy", "Top-5 precision", round(top5_overlap, 3),
                                     0.60, top5_overlap >= 0.60,
                                     f"{int(top5_overlap*5)}/5 overlap"),
    }

accuracy_results = evaluate_matching_accuracy(SYSTEM_SCORES, EXPERT_RANKS, CANDIDATES)

# ── Detailed comparison ──
print("=" * 70)
print("  MATCHING ACCURACY — EXPERT vs SYSTEM RANKINGS")
print("=" * 70)

# Sort by expert rank for display
ranked = sorted(zip(CANDIDATES, EXPERT_RANKS, SYSTEM_SCORES), key=lambda x: x[1])
score_order = np.argsort(SYSTEM_SCORES)[::-1]
sys_ranks = np.empty(len(SYSTEM_SCORES), dtype=int)
sys_ranks[score_order] = np.arange(1, len(SYSTEM_SCORES) + 1)

print(f"  {'Candidate':<20} {'Expert':>8} {'System':>8} {'Score':>8} {'Delta':>8}")
print("  " + "-" * 56)
for name, exp_rank, score in ranked:
    idx = CANDIDATES.index(name)
    s_rank = sys_ranks[idx]
    delta = abs(exp_rank - s_rank)
    marker = "✓" if delta <= 1 else "~" if delta <= 2 else "✗"
    print(f"  {name:<20} {exp_rank:>8} {s_rank:>8} {score:>8.2f} {delta:>6}  {marker}")

print("\n  Rank correlation metrics:")
for key, result in accuracy_results.items():
    status = "✓ PASS" if result.passes else "✗ FAIL"
    print(f"    {result.metric_name:<18} = {result.value:>6.3f}  "
          f"(benchmark: {result.benchmark:.2f})  [{status}]  {result.detail}")

## Section 3 — Rubric consistency

A reliable system should produce nearly identical scores when evaluating the
same resume multiple times. We simulate repeated scoring and measure variance.
Inter-human-rater reliability (Cohen's κ ≈ 0.2 for unstructured interviews)
is our baseline to beat.

In [ ]:
def evaluate_consistency(
    base_scores: List[float],
    n_runs: int = 3,
    noise_std: float = 0.15,
) -> Dict[str, EvalResult]:
    """Evaluate scoring consistency across repeated runs."""
    # Simulate repeated evaluations with small noise
    all_runs: List[List[float]] = [base_scores]
    for _ in range(n_runs - 1):
        noisy = [max(1.0, min(5.0, s + np.random.normal(0, noise_std)))
                 for s in base_scores]
        all_runs.append(noisy)

    runs_array = np.array(all_runs)  # shape: (n_runs, n_candidates)

    # Per-candidate variance
    per_candidate_var = np.var(runs_array, axis=0)
    mean_variance = float(np.mean(per_candidate_var))
    max_variance = float(np.max(per_candidate_var))

    # Coefficient of variation
    per_candidate_cv = np.std(runs_array, axis=0) / np.mean(runs_array, axis=0)
    mean_cv = float(np.mean(per_candidate_cv))

    # Rank stability: how often does the top-3 stay the same?
    top3_sets = [set(np.argsort(run)[-3:]) for run in all_runs]
    rank_stability = sum(
        len(top3_sets[0] & top3_sets[i]) / 3.0
        for i in range(1, n_runs)
    ) / (n_runs - 1)

    return {
        "mean_variance": EvalResult("Consistency", "Mean score variance",
                                    round(mean_variance, 4), 0.05,
                                    mean_variance <= 0.05,
                                    f"Max var: {max_variance:.4f}"),
        "mean_cv": EvalResult("Consistency", "Mean CV",
                              round(mean_cv, 4), 0.10,
                              mean_cv <= 0.10,
                              "Coefficient of variation"),
        "rank_stability": EvalResult("Consistency", "Top-3 rank stability",
                                     round(rank_stability, 3), 0.80,
                                     rank_stability >= 0.80,
                                     f"Across {n_runs} runs"),
    }, runs_array

consistency_results, runs_array = evaluate_consistency(
    SYSTEM_SCORES, config.consistency_runs
)

print("=" * 60)
print("  RUBRIC CONSISTENCY — REPEATED SCORING")
print("=" * 60)
print(f"  Runs: {config.consistency_runs} | Candidates: {len(CANDIDATES)}")
print()
print(f"  {'Candidate':<20} " + "  ".join(f"{'Run '+str(i+1):>7}" for i in range(config.consistency_runs))
      + f"  {'σ':>7}")
print("  " + "-" * (20 + 9 * config.consistency_runs + 9))
for j, name in enumerate(CANDIDATES):
    scores_str = "  ".join(f"{runs_array[i][j]:>7.2f}" for i in range(config.consistency_runs))
    std = np.std(runs_array[:, j])
    print(f"  {name:<20} {scores_str}  {std:>7.3f}")

print("\n  Consistency metrics:")
for key, result in consistency_results.items():
    status = "✓ PASS" if result.passes else "✗ FAIL"
    print(f"    {result.metric_name:<22} = {result.value:>7.4f}  "
          f"(benchmark: {result.benchmark:.2f})  [{status}]")

# ── Compare to human benchmarks ──
print("\n  Comparison to human rater benchmarks:")
print(f"    Unstructured interviews κ ≈ 0.20 (poor)")
print(f"    Structured interviews   κ ≈ 0.60 (substantial)")
print(f"    Our system CV           = {consistency_results['mean_cv'].value:.3f} "
      f"(lower = more consistent)")

## Section 4 — Fairness analysis

We compute three key fairness metrics across demographic groups:

1. **Demographic parity** — Equal pass rates across groups
2. **Equal opportunity** — Equal true-positive rates among qualified candidates
3. **4/5ths rule** — Adverse impact ratio ≥ 0.80 (EEOC standard)

In [ ]:
def evaluate_fairness(
    scores: List[float],
    groups: List[str],
    expert_ranks: List[int],
    threshold: float = 3.0,
    qualified_rank: int = 6,
) -> Dict[str, EvalResult]:
    """Comprehensive fairness evaluation."""
    unique_groups = sorted(set(groups))
    if len(unique_groups) < 2:
        return {}

    g_a, g_b = unique_groups[0], unique_groups[1]
    idx_a = [i for i, g in enumerate(groups) if g == g_a]
    idx_b = [i for i, g in enumerate(groups) if g == g_b]

    scores_a = [scores[i] for i in idx_a]
    scores_b = [scores[i] for i in idx_b]

    # ── Demographic parity: pass rates ──
    pass_a = sum(1 for s in scores_a if s >= threshold) / len(scores_a)
    pass_b = sum(1 for s in scores_b if s >= threshold) / len(scores_b)
    dp_ratio = pass_b / pass_a if pass_a > 0 else 1.0

    # ── Equal opportunity: TPR among qualified ──
    qualified_a = [i for i in idx_a if expert_ranks[i] <= qualified_rank]
    qualified_b = [i for i in idx_b if expert_ranks[i] <= qualified_rank]
    tpr_a = (sum(1 for i in qualified_a if scores[i] >= threshold) /
             len(qualified_a)) if qualified_a else 0
    tpr_b = (sum(1 for i in qualified_b if scores[i] >= threshold) /
             len(qualified_b)) if qualified_b else 0
    eo_ratio = tpr_b / tpr_a if tpr_a > 0 else 1.0

    # ── Mean score gap ──
    mean_a, mean_b = np.mean(scores_a), np.mean(scores_b)
    gap = abs(mean_a - mean_b)

    return {
        "demographic_parity": EvalResult(
            "Fairness", "Demographic parity ratio",
            round(dp_ratio, 3), 0.80, dp_ratio >= 0.80,
            f"Group {g_a}: {pass_a:.1%}, Group {g_b}: {pass_b:.1%}"),
        "equal_opportunity": EvalResult(
            "Fairness", "Equal opportunity ratio",
            round(eo_ratio, 3), 0.80, eo_ratio >= 0.80,
            f"TPR {g_a}: {tpr_a:.1%}, TPR {g_b}: {tpr_b:.1%}"),
        "four_fifths": EvalResult(
            "Fairness", "4/5ths rule (adverse impact)",
            round(min(dp_ratio, 1/dp_ratio if dp_ratio > 0 else 0), 3),
            0.80, min(dp_ratio, 1/dp_ratio if dp_ratio > 0 else 0) >= 0.80,
            "EEOC standard"),
        "score_gap": EvalResult(
            "Fairness", "Mean score gap",
            round(gap, 3), 0.50, gap <= 0.50,
            f"Group {g_a}: {mean_a:.2f}, Group {g_b}: {mean_b:.2f}"),
    }

fairness_results = evaluate_fairness(SYSTEM_SCORES, GROUPS, EXPERT_RANKS)

print("=" * 60)
print("  FAIRNESS ANALYSIS — COMPREHENSIVE REPORT")
print("=" * 60)

# Group breakdown
for g in sorted(set(GROUPS)):
    idxs = [i for i, gr in enumerate(GROUPS) if gr == g]
    g_scores = [SYSTEM_SCORES[i] for i in idxs]
    g_pass = sum(1 for s in g_scores if s >= config.pass_threshold)
    print(f"  Group {g}: n={len(idxs)}, mean={np.mean(g_scores):.2f}, "
          f"pass={g_pass}/{len(idxs)} ({g_pass/len(idxs):.0%})")

print()
for key, result in fairness_results.items():
    status = "✓ PASS" if result.passes else "✗ FAIL"
    print(f"  {result.metric_name:<30} = {result.value:>6.3f}  "
          f"(threshold: {result.benchmark:.2f})  [{status}]")
    print(f"    {result.detail}")

# ── Visualization ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Score distributions by group
for g in sorted(set(GROUPS)):
    g_scores = [SYSTEM_SCORES[i] for i, gr in enumerate(GROUPS) if gr == g]
    axes[0].hist(g_scores, bins=5, alpha=0.6, label=f"Group {g}", edgecolor="white")
axes[0].axvline(x=config.pass_threshold, color="red", linestyle="--", label="Threshold")
axes[0].set_xlabel("Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Score distribution by group")
axes[0].legend()

# Fairness metrics bar chart
metrics = list(fairness_results.values())
names = [m.metric_name.replace(" ", "\n") for m in metrics]
values = [m.value for m in metrics]
colors = ["#388e3c" if m.passes else "#d32f2f" for m in metrics]
axes[1].barh(names, values, color=colors, height=0.5)
axes[1].axvline(x=0.8, color="orange", linestyle="--", label="0.80 threshold")
axes[1].set_xlim(0, 1.5)
axes[1].set_title("Fairness metrics")
axes[1].legend()

plt.tight_layout()
plt.show()
print("\n✓ Fairness report generated")

## Section 5 — Explanation quality

We evaluate explanations using LLM-as-judge across four dimensions:
**specific** (references concrete skills/experience), **evidence-based**
(cites resume content), **actionable** (useful for hiring decision),
and **fair** (no bias indicators).

In [ ]:
JUDGE_PROMPT = """You are an expert in hiring practices. Evaluate this candidate
assessment explanation for quality.

CANDIDATE: {candidate}
EXPLANATION: {explanation}

Score each dimension 1-5:
- specific: Does it reference concrete skills, years, or technologies?
- evidence_based: Does it cite actual resume content?
- actionable: Does it help a hiring manager make a decision?
- fair: Is it free from bias indicators (age, gender, ethnicity, etc.)?

Respond in this exact JSON format:
{{"specific": <int>, "evidence_based": <int>, "actionable": <int>, "fair": <int>}}"""

def evaluate_explanation(candidate: str, explanation: str) -> Dict[str, int]:
    """Evaluate a single explanation using LLM-as-judge."""
    if client:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": JUDGE_PROMPT.format(
                candidate=candidate, explanation=explanation)}],
            temperature=0.0,
            response_format={"type": "json_object"},
        )
        return json.loads(resp.choices[0].message.content)
    else:
        # ── Deterministic mock based on explanation properties ──
        has_numbers = bool(re.search(r"\d+", explanation))
        has_skills = any(s in explanation.lower() for s in
                        ["python", "ml", "pytorch", "sql", "kubernetes"])
        has_evidence = any(w in explanation.lower() for w in
                          ["published", "deployed", "built", "led", "years"])
        has_recommendation = any(w in explanation.lower() for w in
                                ["recommend", "fit", "candidate", "interview"])
        base = 3
        return {
            "specific": min(5, base + has_numbers + has_skills),
            "evidence_based": min(5, base + has_evidence + has_numbers),
            "actionable": min(5, base + has_recommendation + has_evidence),
            "fair": min(5, base + 1),  # assume generally fair
        }

# ── Evaluate all explanations ──
explanation_scores: List[Dict[str, int]] = []
for i, (cand, expl) in enumerate(zip(CANDIDATES, EXPLANATIONS)):
    scores = evaluate_explanation(cand, expl)
    explanation_scores.append(scores)

df_qual = pd.DataFrame(explanation_scores, index=CANDIDATES)

print("=" * 70)
print("  EXPLANATION QUALITY — LLM-AS-JUDGE EVALUATION")
print("=" * 70)
print(df_qual.to_string())

# ── Summary statistics ──
print("\n  Dimension averages:")
dimensions = ["specific", "evidence_based", "actionable", "fair"]
for dim in dimensions:
    mean = df_qual[dim].mean()
    std = df_qual[dim].std()
    bar = "█" * int(mean) + "░" * (5 - int(mean))
    print(f"    {dim:<16} {bar} {mean:.2f} ± {std:.2f}")

overall_quality = df_qual.values.mean()
print(f"\n  Overall explanation quality: {overall_quality:.2f}/5.0")

# ── Visualization ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Per-dimension averages
means = [df_qual[d].mean() for d in dimensions]
colors = ["#1976d2", "#388e3c", "#f57c00", "#7b1fa2"]
axes[0].barh(dimensions, means, color=colors, height=0.5)
axes[0].set_xlim(0, 5.5)
axes[0].set_xlabel("Average score (1-5)")
axes[0].set_title("Explanation quality by dimension")

# Per-candidate overall quality
candidate_means = df_qual.mean(axis=1)
axes[1].barh(CANDIDATES, candidate_means, color="#1976d2", height=0.6)
axes[1].set_xlim(0, 5.5)
axes[1].set_xlabel("Average quality score")
axes[1].set_title("Explanation quality by candidate")

plt.tight_layout()
plt.show()
print("\n✓ Explanation quality evaluation complete")

## Section 6 — Cost and efficiency

We compare the AI recruitment matching engine against manual screening on
time, cost, and quality metrics. The business case rests on maintaining or
improving quality while dramatically reducing cost and time.

In [ ]:
@dataclass
class EfficiencyMetric:
    """Cost/efficiency metric comparison."""
    name: str
    manual_value: float
    ai_value: float
    unit: str
    improvement: str

def compute_efficiency(
    n_candidates: int = 10,
    n_jobs: int = 3,
) -> Tuple[List[EfficiencyMetric], Dict[str, float]]:
    """Compute efficiency metrics for AI vs manual screening."""
    total_evaluations = n_candidates * n_jobs

    # Time estimates
    manual_time_per_eval: float = 23.0 / 10  # 23 hrs per hire, ~10 candidates
    ai_time_per_eval: float = 0.02           # ~1.2 seconds per evaluation

    # Cost estimates
    recruiter_hourly: float = 45.0
    api_cost_per_eval: float = 0.08          # GPT-4o-mini cost

    metrics: List[EfficiencyMetric] = [
        EfficiencyMetric(
            "Time per evaluation",
            manual_time_per_eval * 60, ai_time_per_eval * 60,
            "minutes",
            f"{manual_time_per_eval * 60 / (ai_time_per_eval * 60):.0f}× faster",
        ),
        EfficiencyMetric(
            "Total screening time",
            manual_time_per_eval * total_evaluations,
            ai_time_per_eval * total_evaluations,
            "hours",
            f"{manual_time_per_eval * total_evaluations - ai_time_per_eval * total_evaluations:.1f} hrs saved",
        ),
        EfficiencyMetric(
            "Cost per evaluation",
            manual_time_per_eval * recruiter_hourly,
            api_cost_per_eval,
            "USD",
            f"{manual_time_per_eval * recruiter_hourly / api_cost_per_eval:.0f}× cheaper",
        ),
        EfficiencyMetric(
            "Total screening cost",
            manual_time_per_eval * recruiter_hourly * total_evaluations,
            api_cost_per_eval * total_evaluations,
            "USD",
            f"${manual_time_per_eval * recruiter_hourly * total_evaluations - api_cost_per_eval * total_evaluations:,.0f} saved",
        ),
    ]

    # Aggregate quality metrics
    quality: Dict[str, float] = {
        "rank_correlation": accuracy_results["kendall_tau"].value,
        "consistency_cv": consistency_results["mean_cv"].value,
        "fairness_ratio": fairness_results["demographic_parity"].value,
        "explanation_quality": float(df_qual.values.mean()),
    }

    return metrics, quality

efficiency_metrics, quality_metrics = compute_efficiency()

# ── Display efficiency ──
print("╔" + "═" * 62 + "╗")
print("║          RECRUITMENT ENGINE — EVALUATION SCORECARD          ║")
print("╠" + "═" * 62 + "╣")

print("║  EFFICIENCY                                                  ║")
print("║" + "-" * 62 + "║")
for m in efficiency_metrics:
    print(f"║  {m.name:<24} Manual: {m.manual_value:>10.2f} {m.unit:<6}         ║")
    print(f"║  {'':<24}    AI: {m.ai_value:>10.2f} {m.unit:<6}         ║")
    print(f"║  {'':<24}    → {m.improvement:<30}    ║")

print("║" + " " * 62 + "║")
print("║  QUALITY                                                     ║")
print("║" + "-" * 62 + "║")
quality_benchmarks: Dict[str, Tuple[float, str]] = {
    "rank_correlation": (0.6, "Kendall τ vs expert"),
    "consistency_cv": (0.10, "CV (lower = better)"),
    "fairness_ratio": (0.80, "Demographic parity"),
    "explanation_quality": (3.5, "LLM-judge (1-5)"),
}
for key, value in quality_metrics.items():
    benchmark, label = quality_benchmarks[key]
    if key == "consistency_cv":
        status = "✓" if value <= benchmark else "✗"
    else:
        status = "✓" if value >= benchmark else "✗"
    print(f"║  {status} {label:<28} = {value:>6.3f}  (target: {benchmark:.2f})    ║")

print("║" + " " * 62 + "║")

# ── Overall verdict ──
all_pass = all(r.passes for r in list(accuracy_results.values()) +
               list(consistency_results.values()) + list(fairness_results.values()))
verdict = "PRODUCTION READY" if all_pass else "NEEDS IMPROVEMENT"
print(f"║  VERDICT: {verdict:<50}║")
print("╚" + "═" * 62 + "╝")

print("\n✓ Evaluation complete — all dimensions assessed.")

In [ ]:
# ── Visualise expert vs system rankings ──
score_order = np.argsort(SYSTEM_SCORES)[::-1]
sys_ranks = np.empty(len(SYSTEM_SCORES), dtype=int)
sys_ranks[score_order] = np.arange(1, len(SYSTEM_SCORES) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: expert rank vs system rank
colors_scatter = ["#1976d2" if GROUPS[i] == "A" else "#f57c00" for i in range(len(CANDIDATES))]
axes[0].scatter(EXPERT_RANKS, sys_ranks, c=colors_scatter, s=80, edgecolors="black", zorder=3)
for i, name in enumerate(CANDIDATES):
    axes[0].annotate(name.split()[0], (EXPERT_RANKS[i], sys_ranks[i]),
                     fontsize=7, textcoords="offset points", xytext=(5, 5))
axes[0].plot([1, 10], [1, 10], "k--", alpha=0.3, label="Perfect agreement")
axes[0].set_xlabel("Expert rank")
axes[0].set_ylabel("System rank")
axes[0].set_title(f"Ranking agreement (τ={accuracy_results['kendall_tau'].value:.2f})")
axes[0].legend()

# Bar: evaluation dimensions summary
dim_names = ["Accuracy\n(Kendall τ)", "Consistency\n(1 - CV)", "Fairness\n(DP ratio)", "Explanation\n(quality/5)"]
dim_values = [
    accuracy_results["kendall_tau"].value,
    1 - consistency_results["mean_cv"].value,
    fairness_results["demographic_parity"].value,
    float(df_qual.values.mean()) / 5.0,
]
dim_colors = ["#388e3c" if v >= 0.6 else "#d32f2f" for v in dim_values]
axes[1].bar(dim_names, dim_values, color=dim_colors, width=0.5)
axes[1].set_ylim(0, 1.2)
axes[1].axhline(y=0.8, color="orange", linestyle="--", label="Target (0.80)")
axes[1].set_ylabel("Normalised score")
axes[1].set_title("Evaluation dimensions summary")
axes[1].legend()

plt.tight_layout()
plt.show()
print("✓ Evaluation visualisations rendered")

## Exercises

1. **Threshold optimisation** — Implement a grid search over pass thresholds
   (2.0 to 4.0 in 0.25 steps). For each threshold, compute all fairness metrics
   and plot demographic parity ratio vs threshold. Find the threshold that
   maximises fairness while keeping at least 30 % of candidates passing.

2. **Bootstrapped confidence intervals** — Implement bootstrap resampling
   (1,000 iterations) for each accuracy metric. Report 95 % confidence
   intervals for Kendall τ and Spearman ρ. Are the correlations significantly
   different from zero?

3. **Multi-rubric comparison** — Create three rubric configurations with
   different weight distributions. Evaluate each on accuracy, consistency,
   and fairness. Which configuration produces the best balance across all
   three dimensions?

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | Rank correlation (Kendall τ, Spearman ρ) measures whether the system agrees with expert judgment |
| 2 | Rubric consistency (CV < 0.10) demonstrates the system is more reliable than unstructured interviews |
| 3 | The 4/5ths rule provides a legally-grounded threshold for detecting adverse impact |
| 4 | LLM-as-judge evaluates explanation quality across specificity, evidence, actionability, and fairness |
| 5 | AI screening is 100×+ faster and cheaper than manual review while maintaining quality |
| 6 | Fairness must be evaluated at every threshold — what's fair at 3.0 may not be fair at 4.0 |
| 7 | Production deployment requires all dimensions passing: accuracy, consistency, fairness, and explainability |

## What's Next

You have completed the **AI Recruitment Matching** lab series. You now know how
to build, evaluate, and audit an AI-powered hiring system that is faster, fairer,
and more consistent than traditional ATS keyword matching.

**Potential extensions**:
- Integrate with real ATS APIs (Greenhouse, Lever, Workday)
- Add real-time bias monitoring with alerts
- Build candidate-facing explanations (GDPR Article 22 compliance)
- Implement feedback loops from hiring outcomes

---

## References

1. EEOC, *Uniform Guidelines on Employee Selection Procedures (4/5ths rule)*, 1978 — <https://www.eeoc.gov/>
2. Raghavan, M. et al., *Mitigating Bias in Algorithmic Hiring: Evaluating Claims and Practices*, FAT* 2020.
3. SHRM, *Average Cost-Per-Hire for Companies Is $4,700*, 2023 — <https://www.shrm.org/>
4. EU AI Act, *Regulation (EU) 2024/1689 — Artificial Intelligence Act*, 2024 — <https://artificialintelligenceact.eu/>